# Agri-Master: Fine-tune Faster R-CNN on pest detection dataset

This is the comparison model against YOLOv9 -- Faster R-CNN is a two-stage detector
(region proposals, then classification per region), generally slower but often more
accurate on small/hard objects than one-stage detectors like YOLO.

Unlike Ultralytics YOLO, torchvision's Faster R-CNN has no built-in `.train()` method --
we write the training loop ourselves below, which is worth seeing explicitly once.

Same setup as the YOLOv9 notebook:
1. **Runtime -> Change runtime type -> T4 GPU**, then Save.
2. Add your `ROBOFLOW_API_KEY` as a Colab Secret (key icon in the left sidebar).

In [ ]:
!pip install -q roboflow torchmetrics

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
!nvidia-smi

In [ ]:
ROBOFLOW_API_KEY = None
try:
    from google.colab import userdata
    ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
except Exception:
    pass

if not ROBOFLOW_API_KEY:
    from getpass import getpass
    ROBOFLOW_API_KEY = getpass("Enter your Roboflow API key: ")

## Download and filter the dataset (identical to the YOLOv9 notebook)

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("pest-2bk0e").project("detection-d0qov")
version = project.version(1)
dataset = version.download("yolov8", location="/content/raw_dataset")
print(dataset.location)

In [ ]:
import shutil
from pathlib import Path

import yaml

RAW_DIR = Path("/content/raw_dataset")
OUT_DIR = Path("/content/dataset_15")

KEEP_CLASSES = [
    "Cicadellidae", "aphids", "Miridae", "blister beetle", "mole cricket",
    "grub", "Locustoidea", "wireworm", "Unaspis yanonensis",
    "legume blister beetle", "flea beetle", "flax budworm", "Prodenia litura",
    "beet army worm", "corn borer",
]

with open(RAW_DIR / "data.yaml") as f:
    raw_names = yaml.safe_load(f)["names"]

old_to_new = {i: KEEP_CLASSES.index(n) for i, n in enumerate(raw_names) if n in KEEP_CLASSES}
assert len(old_to_new) == len(KEEP_CLASSES)

kept, dropped = 0, 0
for split in ["train", "valid", "test"]:
    src_images, src_labels = RAW_DIR / split / "images", RAW_DIR / split / "labels"
    dst_images, dst_labels = OUT_DIR / split / "images", OUT_DIR / split / "labels"
    dst_images.mkdir(parents=True, exist_ok=True)
    dst_labels.mkdir(parents=True, exist_ok=True)

    for label_path in src_labels.glob("*.txt"):
        lines = []
        for line in label_path.read_text().splitlines():
            if not line.strip():
                continue
            parts = line.split()
            old_idx = int(parts[0])
            if old_idx in old_to_new:
                parts[0] = str(old_to_new[old_idx])
                lines.append(" ".join(parts))
        if not lines:
            dropped += 1
            continue
        (dst_labels / label_path.name).write_text("\n".join(lines) + "\n")
        matches = list(src_images.glob(f"{label_path.stem}.*"))
        if matches:
            shutil.copy(matches[0], dst_images / matches[0].name)
            kept += 1

print(f"Kept {kept} images, dropped {dropped}")

## Dataset class: converts YOLO-format labels (normalized center/width/height)
into the [xmin, ymin, xmax, ymax] pixel format Faster R-CNN expects.

Note: Faster R-CNN reserves class 0 for "background", so our label ids shift by +1.

In [ ]:
from PIL import Image
from torch.utils.data import Dataset
from torchvision.transforms import functional as TF


class YoloDetectionDataset(Dataset):
    def __init__(self, images_dir, labels_dir):
        self.images_dir = Path(images_dir)
        self.labels_dir = Path(labels_dir)
        self.image_files = sorted(self.images_dir.glob("*.jpg"))

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = self.image_files[idx]
        img = Image.open(img_path).convert("RGB")
        w, h = img.size

        label_path = self.labels_dir / f"{img_path.stem}.txt"
        boxes, labels = [], []
        for line in label_path.read_text().splitlines():
            cls, xc, yc, bw, bh = map(float, line.split())
            xc, yc, bw, bh = xc * w, yc * h, bw * w, bh * h
            boxes.append([xc - bw / 2, yc - bh / 2, xc + bw / 2, yc + bh / 2])
            labels.append(int(cls) + 1)  # +1: class 0 is background

        target = {
            "boxes": torch.as_tensor(boxes, dtype=torch.float32),
            "labels": torch.as_tensor(labels, dtype=torch.int64),
            "image_id": torch.tensor([idx]),
        }
        return TF.to_tensor(img), target


def collate_fn(batch):
    return tuple(zip(*batch))

In [ ]:
from torch.utils.data import DataLoader

train_dataset = YoloDetectionDataset(OUT_DIR / "train/images", OUT_DIR / "train/labels")
valid_dataset = YoloDetectionDataset(OUT_DIR / "valid/images", OUT_DIR / "valid/labels")
test_dataset = YoloDetectionDataset(OUT_DIR / "test/images", OUT_DIR / "test/labels")

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=2, collate_fn=collate_fn)

print(f"train={len(train_dataset)} valid={len(valid_dataset)} test={len(test_dataset)}")

## Build the model

Starts from COCO-pretrained weights (transfer learning, same idea as YOLOv9c.pt),
then we swap the classification head for our 15 classes + background.
`min_size`/`max_size` are capped below torchvision's defaults (800/1333) to keep
training time reasonable on a free Colab GPU.

In [ ]:
from torchvision.models.detection import FasterRCNN_ResNet50_FPN_Weights, fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

NUM_CLASSES = len(KEEP_CLASSES) + 1  # +1 background
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = fasterrcnn_resnet50_fpn(
    weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT,
    min_size=480,
    max_size=640,
)
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)
model.to(device)
print("Model ready on", device)

## Train

10 epochs (fewer than YOLOv9's 50) -- Faster R-CNN is heavier per-image, so this is
a deliberate time/accuracy tradeoff for a free-tier GPU. The loss printed is the sum
of the model's internal losses (box regression + classification + region proposal).

In [ ]:
EPOCHS = 10

optimizer = torch.optim.SGD(model.parameters(), lr=0.005, momentum=0.9, weight_decay=0.0005)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    for images, targets in train_loader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        total_loss += losses.item()

    lr_scheduler.step()
    print(f"Epoch {epoch + 1}/{EPOCHS} - avg loss: {total_loss / len(train_loader):.4f}")

## Evaluate on the held-out test set (same mAP metric as the YOLOv9 notebook, for a fair comparison)

In [ ]:
from torchmetrics.detection.mean_ap import MeanAveragePrecision

metric = MeanAveragePrecision()
model.eval()
with torch.no_grad():
    for images, targets in test_loader:
        images = [img.to(device) for img in images]
        preds = model(images)
        preds = [{k: v.cpu() for k, v in p.items()} for p in preds]
        targets_cpu = [{"boxes": t["boxes"], "labels": t["labels"]} for t in targets]
        metric.update(preds, targets_cpu)

results = metric.compute()
print("mAP50-95:", results["map"].item())
print("mAP50:", results["map_50"].item())
print("mAR (100 dets/img):", results["mar_100"].item())

## Save and download the trained weights

In [ ]:
torch.save(model.state_dict(), "/content/pest_fasterrcnn_best.pt")

try:
    from google.colab import files
    files.download("/content/pest_fasterrcnn_best.pt")
except Exception:
    print("Not running in Colab - file saved at /content/pest_fasterrcnn_best.pt")